<a href="https://colab.research.google.com/github/imdoamaral/formacao-pnl/blob/main/supervisionado_vs_regras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Pré-processamento

In [1]:
import pandas as pd
from google.colab import files
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
# Selecionar o arquivo Tweets2.csv
files.upload()

In [3]:
Tweets = pd.read_csv('Tweets2.csv')

In [4]:
Tweets.shape

(74682, 4)

In [5]:
Tweets.head()

,id,local,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [6]:
Tweets.groupby(['sentiment']).size()

,0
sentiment,
Irrelevant,12990
Negative,22542
Neutral,18318
Positive,20832


In [8]:
Tweets.loc[Tweets['sentiment']=='Irrelevant', 'sentiment'] = 'Neutral'

In [9]:
Tweets.groupby(['sentiment']).size()

,0
sentiment,
Negative,22542
Neutral,31308
Positive,20832


In [10]:
Tweets = Tweets.dropna(subset=['text'])
Tweets.reset_index(drop=True, inplace=True)

In [11]:
Tweets.shape

(73996, 4)

# Supervisionado

In [12]:
# Transforma palavras em números

token = Tokenizer(num_words=100)  # Mantem apenas as 100 palavras mais frequentes
token.fit_on_texts(Tweets['text'].values)  # Mapeia cada palavra a um numero inteiro unico

In [13]:
# 'Padding' = completa o texto pra ele ficar de forma tabular

X = token.texts_to_sequences(Tweets['text'].values)  # Substitui as palavras por números
X = pad_sequences(X, padding='post', maxlen=100) # Todas as entradas no mesmo tamanho

In [15]:
# Transforma os sentimentos em números

labelencoder = LabelEncoder()
y = labelencoder.fit_transform(Tweets['sentiment'])
print(y)

[2 2 2 ... 2 2 2]


In [16]:
# One-Hot Encoding = transforma rótulos de categoria em uma matriz binaria

y = to_categorical(y)
print(y)

[[0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]
 ...
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]]


In [17]:
# Divide os dados em 2 grupos: um para o computador aprender e outro para testarmos
# se ele realmente aprendeu.
# - X = texto tokenizado
# - y = variáveis categóricas

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.4) # 40% para teste
X_test

array([[10, 13,  4, ...,  0,  0,  0],
       [12,  5, 57, ...,  0,  0,  0],
       [ 1,  7,  4, ...,  0,  0,  0],
       ...,
       [10,  7, 53, ...,  0,  0,  0],
       [ 2, 29,  9, ...,  0,  0,  0],
       [54,  4, 54, ...,  0,  0,  0]], dtype=int32)

In [18]:
# --- Inicia a construçao da rede neural ---

modelo = Sequential() # modelo vazio para empilhar camadas uma apos a outra

# Transforma numeros (indices de palavras) em vetores densos
# input_dim = tamanho do vocabulario
# 'outuput_dim = 128' = cada palavra sera representada por um vetor de 128 numeros
# input_length = indica que cada entrada tem 100 palavras (tamanho definido no padding)
modelo.add(Embedding(input_dim=len(token.word_index),
                     output_dim=128,
                     ))

# Descarta aleatoriamente 20% de colunas inteiras de dados durante o treinamento
# Isso ajuda a evitar que o modelo decore padroes especificos demais (overfitting)
modelo.add(SpatialDropout1D(0.2))

# É a memória da rede neural
# As LSTMs conseguem lembrar o contexto de palavras que apareceram no inicio da frase
# para entender o sentido final.
modelo.add(LSTM(units=196,  # numero de neuronios internos
                dropout=0.2,  # outra tecnica contra o vicio nos dados de treino
                recurrent_dropout=0,
                activation = 'tanh',  # funçao matematica que processa a informaçao
                recurrent_activation = 'sigmoid',
                unroll = False,
                use_bias = True
                ))
modelo.add(Dense(units=3,  # representa as 3 categorias (negativo, neutro, positivo)
                 activation='softmax' # transforma a saida em probabilidades
                 ))


In [19]:
# Cria o modelo

modelo.compile(loss='categorical_crossentropy',  # funçao que mede o erro do modelo
               optimizer='adam',  # o motor que ajusta os pesos da rede
               metrics=['accuracy']  # define que queremos ver a % de acertos
               )

print(modelo.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [20]:
# Informamos o formato da entrada (batch_size=None, sequence_length=100)
# Isso força o Keras a construir os pesos para que possamos ver o Param #
modelo.build(input_shape=(None, 100))
print(modelo.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     4,324,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 100, 128)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 196)            │       254,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,579,615 (17.47 MB)

 Trainable params: 4,579,615 (17.47 MB)

 Non-trainable params: 0 (0.00 B)

None


In [21]:
# --- Treinando o modelo da rede neural ---

modelo.fit(X_train,
           y_train,
           epochs=5,  # passará por todo o conjunto de dados de treinamento 10x
           batch_size=500,  # nao processa todos os dados de uma vez
           verbose=True
           )

Epoch 1/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.4180 - loss: 1.0832
Epoch 2/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.4194 - loss: 1.0826
Epoch 3/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 182s 2s/step - accuracy: 0.4194 - loss: 1.0822
Epoch 4/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 180s 2s/step - accuracy: 0.4194 - loss: 1.0826
Epoch 5/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.4194 - loss: 1.0823


In [22]:
_, accuracy = modelo.evaluate(X_test, y_test)
print('Accuracy:', accuracy)

925/925 ━━━━━━━━━━━━━━━━━━━━ 83s 89ms/step - accuracy: 0.4176 - loss: 1.0831
Accuracy: 0.41761547327041626


# Vader (Regras)

In [23]:
mas = SentimentIntensityAnalyzer()
Tweets['vader_sentiment'] = ''

for y in range(len(Tweets.index)):
  x = mas.polarity_scores(Tweets['text'].iloc[y]) # analisa o texto da linha y
  del x['compound']
  maior = max(x, key=x.get)  # verifica qual das chaves possui o maior valor
  Tweets.loc[y, 'vader_sentiment'] = maior

In [25]:
# Faz a mesma coisa que o código anterior, porém de forma mais rápida (otimizada)

def get_vader(text):
    scores = mas.polarity_scores(text)
    del scores['compound']
    return max(scores, key=scores.get)

Tweets['vader_sentiment'] = Tweets['text'].apply(get_vader)

In [26]:
Tweets.groupby(['vader_sentiment']).size()

,0
vader_sentiment,
neg,3660
neu,65581
pos,4755


In [27]:
Tweets.groupby(['sentiment']).size()

,0
sentiment,
Negative,22358
Neutral,30983
Positive,20655


In [28]:
Tweets.loc[Tweets['vader_sentiment'] == 'neu', 'vader_sentiment'] = 'Neutral'
Tweets.loc[Tweets['vader_sentiment'] == 'neg', 'vader_sentiment'] = 'Negative'
Tweets.loc[Tweets['vader_sentiment'] == 'pos', 'vader_sentiment'] = 'Positive'

In [29]:
Tweets.groupby(['vader_sentiment']).size()

,0
vader_sentiment,
Negative,3660
Neutral,65581
Positive,4755


In [30]:
y_pred = Tweets['vader_sentiment']
y_test = Tweets['sentiment']

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[ 2004 19902   452]
 [ 1122 28384  1477]
 [  534 17295  2826]]


In [31]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.44886210065408944


**Conclusão:** o modelo de regras teve um desempenho levemente superior ao LSTM.